# 05 · SHAP Explainability — SR 11-7 Model Validation

**Project 8 / 10**  
SHAP (SHapley Additive exPlanations) for the Random Forest volatility model.  
SR 11-7 model inventory, challenger benchmarking, and feature stability across regimes.

> Author: Niraj Neupane | github.com/nirajneupane17

In [ ]:
import sys; sys.path.insert(0,"../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
import shap
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from feature_engineering import build_vol_features, train_test_split_temporal
from shap_validation import SHAPValidator, default_inventory, challenger_comparison, print_validation_report
print("Libraries loaded")

In [ ]:
rets = pd.read_csv("../data/returns.csv", index_col="Date", parse_dates=True)
spy  = rets["SPY"].dropna()
feat = build_vol_features(spy, windows=[5,10,21,63], target_horizon=21)
X_cols = ["vol_5d","vol_10d","vol_21d","vol_63d","skewness_21d","kurtosis_21d",
          "return_sq","abs_return","vol_ratio","momentum_5d","momentum_21d"]
X_tr, X_te, y_tr, y_te = train_test_split_temporal(feat, 0.20, "vol_target")
X_tr_f = X_tr[X_cols]; X_te_f = X_te[X_cols]
print(f"Train/Test: {X_tr_f.shape} / {X_te_f.shape}")

In [ ]:
# Fit RF and compute SHAP
rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_tr_f, y_tr)
shap_val = SHAPValidator(rf, feature_names=X_cols, model_name="RF Vol Forecaster")
shap_val.fit(X_te_f, max_samples=300)
print("SHAP fitted.")
print(f"Top 5 features:")
print(shap_val.top_features(5).to_string())
print(f"\nTop-3 concentration: {shap_val.concentration_ratio(3)*100:.1f}%")

In [ ]:
# Plot SHAP importance
fig, ax = plt.subplots(figsize=(10,6))
shap_val.plot_importance(ax=ax, top_n=11)
plt.tight_layout(); plt.show()

In [ ]:
img = plt.imread("../results/05_shap_waterfall.png")
fig, ax = plt.subplots(figsize=(15,6)); ax.imshow(img); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# SR 11-7 Model Inventory
inv = default_inventory()
print("SR 11-7 Model Inventory:")
print(inv[["name","risk_tier","validation_status","auc","rmse"]].to_string(index=False))

In [ ]:
# Challenger comparison (vol models)
garch_rmse = float(feat.loc[X_te.index,"vol_21d"].values[:len(y_te)].std())
rf_rmse    = float(np.sqrt(mean_squared_error(y_te, rf.predict(X_te_f))))
ch_res = {"GARCH Baseline":{"rmse":garch_rmse},"Random Forest":{"rmse":rf_rmse}}
ch = challenger_comparison(ch_res)
print("Challenger Comparison:")
print(ch.to_string(index=False))

In [ ]:
# Full validation report
garch_bt = None  # not available in this notebook
print_validation_report(shap_val)